# Apache Iceberg com Apache Spark (PySpark)

## Cenário: Plataforma de E-commerce

Neste notebook, demonstramos o uso do **Apache Iceberg** com **Apache Spark (PySpark)** aplicado ao mesmo sistema de pedidos de e-commerce do notebook Delta Lake.

O cenário modela três entidades:
- **Cliente** — dados cadastrais do comprador
- **Produto** — catálogo com categoria e preço
- **Pedido** — relaciona cliente e produto com status, quantidade e data

Demonstraremos as operações **DDL**, **INSERT**, **UPDATE**, **DELETE** e **Snapshots / Time Travel**.

> **Nota:** Na primeira execução, o Spark baixará o JAR do Iceberg (~35 MB) do Maven Central. Requer conexão à internet.

## Modelo Entidade-Relacionamento

```
┌──────────────┐         ┌──────────────┐
│   CUSTOMER   │         │   PRODUCT    │
├──────────────┤         ├──────────────┤
│ customer_id  │PK       │ product_id   │PK
│ name         │         │ name         │
│ email        │         │ category     │
│ city         │         │ price        │
└──────┬───────┘         └──────┬───────┘
       │                        │
       └──────────┬─────────────┘
                  │
                  ▼
        ┌─────────────────────┐
        │        ORDER        │
        ├─────────────────────┤
        │ order_id     (PK)   │
        │ customer_id  (FK)   │
        │ product_id   (FK)   │
        │ quantity            │
        │ unit_price          │
        │ status              │
        │ order_date          │
        └─────────────────────┘
```

| Entidade | Atributos |
|---|---|
| CUSTOMER | customer_id (PK), name, email, city |
| PRODUCT  | product_id (PK), name, category, price |
| ORDER    | order_id (PK), customer_id (FK), product_id (FK), quantity, unit_price, status, order_date |

## 1. Configuração — SparkSession com Apache Iceberg

O Iceberg não possui pacote pip para o runtime Spark. O JAR é carregado via `spark.jars.packages` diretamente do Maven Central.

Usamos um catálogo nomeado `local` do tipo `hadoop` para armazenamento local.

In [1]:
import sys
import os
# Adiciona o diretório src ao path para permitir importação do pacote local
sys.path.append(os.path.abspath("../src"))

from spark_delta_apache import get_spark_iceberg_session, setup_logging

# Configurar logging e criar SparkSession via pacote local
setup_logging()
spark = get_spark_iceberg_session("IcebergEcommerce")
spark


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-54232b9f-e696-4d09-85e8-bee6c1e7f4c7;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
:: resolution report :: resolve 196ms :: artifacts dl 7ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-subm

## 2. DDL — Criação das Tabelas

As tabelas Iceberg são prefixadas com o nome do catálogo (`local.`), usando `USING iceberg`.
Os metadados ficam em arquivos JSON na pasta `metadata/` de cada tabela.

In [ ]:
# Limpar tabelas existentes para garantir execução limpa
spark.sql("DROP TABLE IF EXISTS local.customer_iceberg")
spark.sql("DROP TABLE IF EXISTS local.product_iceberg")
spark.sql("DROP TABLE IF EXISTS local.order_iceberg")

spark.sql("""
    CREATE TABLE local.customer_iceberg (
        customer_id INT,
        name        STRING,
        email       STRING,
        city        STRING
    ) USING iceberg
""")

spark.sql("""
    CREATE TABLE local.product_iceberg (
        product_id INT,
        name       STRING,
        category   STRING,
        price      DOUBLE
    ) USING iceberg
""")

spark.sql("""
    CREATE TABLE local.order_iceberg (
        order_id    INT,
        customer_id INT,
        product_id  INT,
        quantity    INT,
        unit_price  DOUBLE,
        status      STRING,
        order_date  DATE
    ) USING iceberg
""")

print("Tabelas criadas com sucesso!")
print("  - local.customer_iceberg")
print("  - local.product_iceberg")
print("  - local.order_iceberg")

## 3. INSERT — Inserção de Dados

Cada `INSERT` cria um novo **snapshot** no Iceberg, registrado nos metadados JSON da tabela.

In [ ]:
# Inserir clientes
spark.sql("""
    INSERT INTO local.customer_iceberg VALUES
        (1, 'Ana Silva',   'ana@email.com',   'São Paulo'),
        (2, 'Bruno Costa', 'bruno@email.com', 'Rio de Janeiro'),
        (3, 'Carla Lima',  'carla@email.com', 'Curitiba')
""")

# Inserir produtos
spark.sql("""
    INSERT INTO local.product_iceberg VALUES
        (1, 'Notebook',      'Eletronicos', 3500.00),
        (2, 'Mouse',         'Eletronicos',   89.90),
        (3, 'Cadeira Gamer', 'Moveis',      1200.00)
""")

# Inserir pedidos
spark.sql("""
    INSERT INTO local.order_iceberg VALUES
        (1, 1, 1, 1, 3500.00, 'pendente', DATE '2024-01-10'),
        (2, 2, 2, 2,   89.90, 'aprovado', DATE '2024-01-11'),
        (3, 3, 3, 1, 1200.00, 'pendente', DATE '2024-01-12')
""")

print("=== Clientes ===")
spark.sql("SELECT * FROM local.customer_iceberg").show()

print("=== Produtos ===")
spark.sql("SELECT * FROM local.product_iceberg").show()

print("=== Pedidos ===")
spark.sql("SELECT * FROM local.order_iceberg").show()

## 4. UPDATE — Atualização de Dados

O Iceberg suporta `UPDATE` via **Merge-on-Read** ou **Copy-on-Write** (configurável). Por padrão no modo `hadoop`, usa copy-on-write: novos arquivos são escritos com os dados atualizados e um novo snapshot é criado.

In [4]:
# Atualizar status do pedido 1 para 'aprovado'
spark.sql("""
    UPDATE local.order_iceberg
    SET status = 'aprovado'
    WHERE order_id = 1
""")

# Atualizar preço do Notebook com desconto
spark.sql("""
    UPDATE local.product_iceberg
    SET price = 3299.00
    WHERE product_id = 1
""")

print("=== Pedidos após UPDATE ===")
spark.sql("SELECT * FROM local.order_iceberg").show()

print("=== Produtos após UPDATE ===")
spark.sql("SELECT * FROM local.product_iceberg").show()

=== Pedidos após UPDATE ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       2|          2|         2|       2|      89.9|aprovado|2024-01-11|
|       3|          3|         3|       1|    1200.0|pendente|2024-01-12|
|       1|          1|         1|       1|    3500.0|aprovado|2024-01-10|
+--------+-----------+----------+--------+----------+--------+----------+

=== Produtos após UPDATE ===
+----------+-------------+-----------+------+
|product_id|         name|   category| price|
+----------+-------------+-----------+------+
|         1|     Notebook|Eletronicos|3299.0|
|         2|        Mouse|Eletronicos|  89.9|
|         3|Cadeira Gamer|     Moveis|1200.0|
+----------+-------------+-----------+------+



## 5. DELETE — Remoção de Dados

In [5]:
# Cancelar (remover) o pedido 3
spark.sql("""
    DELETE FROM local.order_iceberg
    WHERE order_id = 3
""")

print("=== Pedidos após DELETE ===")
spark.sql("SELECT * FROM local.order_iceberg").show()

=== Pedidos após DELETE ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       2|          2|         2|       2|      89.9|aprovado|2024-01-11|
|       1|          1|         1|       1|    3500.0|aprovado|2024-01-10|
+--------+-----------+----------+--------+----------+--------+----------+



## 6. MERGE INTO — Upsert de Dados

O `MERGE INTO` é o principal diferencial do Iceberg: permite **upsert** em uma única operação atômica.

- `WHEN MATCHED` — atualiza registros que existem na tabela alvo
- `WHEN NOT MATCHED` — insere registros novos da fonte

Cada `MERGE` gera um novo snapshot, preservando o histórico completo.

In [ ]:
# Fonte de dados para upsert:
# - order_id 1: já existe → atualiza status para 'entregue'
# - order_id 4: não existe → insere novo pedido
spark.sql("""
    MERGE INTO local.order_iceberg AS target
    USING (
        SELECT 1 AS order_id, 1 AS customer_id, 1 AS product_id,
               1 AS quantity, 3500.00 AS unit_price, 'entregue' AS status,
               DATE '2024-01-10' AS order_date
        UNION ALL
        SELECT 4, 1, 2, 3, 89.90, 'aprovado', DATE '2024-01-15'
    ) AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN
        UPDATE SET status = source.status
    WHEN NOT MATCHED THEN
        INSERT *
""")

print("=== Pedidos após MERGE INTO ===")
spark.sql("SELECT * FROM local.order_iceberg ORDER BY order_id").show()

## 7. Snapshots, Histórico e Time Travel

O Iceberg registra cada operação como um **snapshot** imutável. Os metadados ficam em arquivos JSON na pasta `metadata/` da tabela.

Diferente do Delta Lake que usa versões numéricas, o Iceberg usa **snapshot IDs** (números longos) e timestamps.

In [ ]:
# Listar todos os snapshots da tabela order_iceberg
print("=== Snapshots (order_iceberg) ===")
spark.sql("""
    SELECT
        snapshot_id,
        committed_at,
        operation,
        summary['added-records']   AS added_records,
        summary['deleted-records'] AS deleted_records,
        summary['total-records']   AS total_records
    FROM local.order_iceberg.snapshots
    ORDER BY committed_at
""").show(truncate=False)

In [7]:
# Ver histórico de operações
print("=== Histórico de Operações (order_iceberg) ===")
spark.sql("""
    SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor
    FROM local.order_iceberg.history
""").show(truncate=False)

=== Histórico de Operações (order_iceberg) ===
+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-04-28 13:48:06.863|4932698636970967954|NULL               |true               |
|2026-04-28 13:48:11.063|105007949562930770 |4932698636970967954|true               |
|2026-04-28 13:48:12.195|7352906881438807714|105007949562930770 |true               |
+-----------------------+-------------------+-------------------+-------------------+



In [8]:
# Time Travel: ler o estado no primeiro snapshot (após o INSERT inicial)
first_snapshot_id = (
    spark.sql("""
        SELECT snapshot_id
        FROM local.order_iceberg.snapshots
        ORDER BY committed_at
        LIMIT 1
    """)
    .first()["snapshot_id"]
)

print(f"Primeiro snapshot ID: {first_snapshot_id}")
print("=== Time Travel — Primeiro Snapshot (após INSERT) ===")
spark.read.format("iceberg") \
    .option("snapshot-id", first_snapshot_id) \
    .load("local.order_iceberg") \
    .show()

# Estado atual
print("=== Estado Atual ===")
spark.sql("SELECT * FROM local.order_iceberg").show()

Primeiro snapshot ID: 4932698636970967954
=== Time Travel — Primeiro Snapshot (após INSERT) ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       1|          1|         1|       1|    3500.0|pendente|2024-01-10|
|       2|          2|         2|       2|      89.9|aprovado|2024-01-11|
|       3|          3|         3|       1|    1200.0|pendente|2024-01-12|
+--------+-----------+----------+--------+----------+--------+----------+

=== Estado Atual ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       2|          2|         2|       2|      89.9|aprovado|2024-01-11|
|       1|          1|         1|       1|    3500.0|aprovado|2024-0

In [9]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
